# DDQN Kaggle v1: ganancia pre/post

300 camas, 40 días, base y actualización de 7 días. Un agente con pesos predictivos y de ganancia unitarios; λ=0.025.

Días 1–7: ajuste; decisiones 8–40; reward liberado 9–41. Train/validation/test separados por paciente. `plots` usa solamente CSV. Ver README para ejecutar y reanudar.

## Configuración y objetivo causal

In [ ]:
import os, json, time, random, hashlib, platform, traceback, itertools, gzip, pickle, shutil, signal, subprocess
from pathlib import Path
from collections import deque
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, auc
from xgboost import XGBClassifier
import xgboost, sklearn, torch
from torch import nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
SEGMENT_VERSION = 'kaggle-v8'
CODE_ID = '6d13fe8345092a093a3e34df42fc3c78e1487c6a2a8e322cf687bf96f9c29aaa'
MODE = os.environ.get('DARL_DUMMY_MODE', 'full')
if MODE not in ('full','pilot','plots'):
    raise ValueError('Use full, pilot, or CSV-only plots')
SMOKE = False
VITALS = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp']
SCENARIOS = ('stable','covariate','concept','combined')
ACTIONS = ('A1','A2','A3','A4')
BEDS, DAYS, WARMUP, UPDATE_DAYS, LABEL_DELAY = 300, 40, 7, 7, 1
MAX_EPISODES = 30 if MODE == 'pilot' else 1000
MIN_EPISODES = MAX_EPISODES
LR_START, LR_END, EPS_DECAY, EPS_MIN = 5e-4, 5e-5, .99, .05
COST_LAMBDA, Q_TIE, SELECT_WINDOW = .025, .005, 3
ONSET, CONCEPT_DELAY, CONCEPT_MAX, SCALE_GAIN = 8, 2, 3., .4
COV_RATE_GRID, CONCEPT_RATE_GRID = (.25,.35,.5), (.1,.15,.2)
INJECTOR_VERSION = 'temporal_advance_onset_v2'
TRAIN_CYCLE = ('stable','covariate','concept','combined','covariate','concept','combined')
OBS_NAMES = ('ks','psi','auc_drop','ap','valid')
OBS_DIM = 15
OUT = Path(os.environ.get('DARL_DUMMY_OUTPUT', '/kaggle/working/ddqn-v8' if Path('/kaggle/input').exists() else 'kaggle/dqn-dummy/v8/outputs/full'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if MODE != 'plots' and (not Path('/kaggle/input').exists() or DEVICE.type != 'cuda'):
    raise RuntimeError('Experimental execution requires Kaggle and CUDA; plots is CSV-only')
torch.set_num_threads(2)
RUN_STARTED = time.perf_counter()
RUN_BUDGET_SECONDS = float(os.environ.get('DARL_DUMMY_BUDGET_SECONDS','36000'))
STOP_REQUESTED = False
RUN_STATE = {'protocol':'kaggle-v8-prepost-7d-mixed', 'mode':MODE}
APPEND_NAMES = {'train_daily_metrics','train_actions','validation_daily_metrics',
    'validation_actions','validation_history','controller_freeze','actions','daily_metrics'}
progress = dict(version=SEGMENT_VERSION,status='running',mode=MODE,seed=SEED,beds=BEDS,
    days=DAYS,initial_days=WARMUP,update_days=UPDATE_DAYS,history_days=3,
    episodes=MAX_EPISODES,device=str(DEVICE),xgboost_device='cpu',
    python=platform.python_version(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,numpy=np.__version__,pandas=pd.__version__ ,
    sklearn=sklearn.__version__,xgboost=xgboost.__version__,torch=torch.__version__)

@dataclass(frozen=True)
class RewardConfig:
    """Absolute performance and same-record improvement weights."""
    alpha: float = 1.
    beta: float = 1.
    cost_lambda: float = .025
    eta_auc: float = 1.
    eta_ap: float = 1.

    def __post_init__(self):
        """Reject invalid objective coefficients."""
        values = list(asdict(self).values())
        if not np.isfinite(values).all() or min(values) < 0:
            raise ValueError('Reward weights must be finite and nonnegative')

    @property
    def config_id(self):
        """Identify all five weights in exported artifacts."""
        return f'a{self.alpha:g}_b{self.beta:g}_ea{self.eta_auc:g}_ep{self.eta_ap:g}_l{self.cost_lambda:g}'.replace('.','p')

COMMON_CONFIG = RewardConfig()
CONFIGS = (COMMON_CONFIG,)

def reward_components(post, pre, seconds, cost_unit, config):
    """Score a chosen action against its unchanged pipeline on identical daily rows."""
    if not np.isfinite([seconds,cost_unit]).all() or seconds < 0 or cost_unit <= 0:
        raise ValueError('Invalid action time or A4 time reference')
    valid = bool(post['auc_valid'] and pre['auc_valid'] and np.isfinite(
        [post['auc'],post['average_precision'],pre['auc'],pre['average_precision']]).all())
    da = post['auc']-pre['auc'] if valid else np.nan
    dp = post['average_precision']-pre['average_precision'] if valid else np.nan
    ga, gp = config.eta_auc*da, config.eta_ap*dp
    gain = ga+gp
    ac = config.alpha*post['auc'] if valid else 0.
    apc = config.beta*post['average_precision'] if valid else 0.
    relative = seconds/cost_unit
    penalty = config.cost_lambda*relative
    total = ac+apc+(gain if valid else 0.)-penalty
    return dict(predictive_valid=valid,gain_evaluable=valid,delta_auc=da,delta_ap=dp,
        gain_auc_component=ga,gain_ap_component=gp,gain=gain,auc_component=ac,
        ap_component=apc,predictive_utility=ac+apc,cost_relative=relative,
        cost_penalty=penalty,reward=total,common_reward=total,common_cost_penalty=penalty)

class SegmentYield(RuntimeError):
    """Indicate a durable partial run requiring continuation."""

def request_stop(signum, frame):
    """Ask for a checkpoint without interrupting a partially written unit."""
    global STOP_REQUESTED
    STOP_REQUESTED = True

if MODE != 'plots':
    for folder in (OUT,OUT/'models',OUT/'predictions',OUT/'figures'):
        folder.mkdir(parents=True,exist_ok=True)
    signal.signal(signal.SIGUSR1,request_stop) if hasattr(signal,'SIGUSR1') else None


## Utilidades heredadas de v7

In [ ]:
def checkpoint(stage, **values):
    """Persist progress as CSV after each costly operation."""
    progress.update(stage=stage, **values)
    (OUT / 'run_summary.json').write_text(json.dumps(progress, indent=2, default=str), encoding='utf-8')
    table("run_summary", pd.DataFrame([{"key": k, "value": str(v)} for k, v in progress.items()]))
    print(
        f"[{stage}]",
        {
            k: v
            for k, v in values.items()
            if k in ("episodes_done", "status", "selected_episode")
        },
        flush=True,
    )

def table(name, rows):
    """Persist one inspectable table without JSON outputs."""
    data = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    destination = OUT / f"{name}.csv"
    temporary = destination.with_suffix(".csv.pending")
    data.to_csv(temporary, index=False, float_format="%.17g")
    os.replace(temporary, destination)
    return data

def reset_seed():
    """Restart every independent agent with seed 42 and the same initialization."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def append_table(name, rows):
    """Append a completed batch, avoiding repeated rewrites of trajectory CSVs."""
    data = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    if len(data):
        path = OUT / f"{name}.csv"
        data.to_csv(path, mode="a", header=not path.exists(), index=False, float_format="%.17g")

def encode_observation(metrics, last_auc, last_ap, auc_reference, ap_reference):
    """Map five causal observations continuously to [0,1], without fitted scalers."""
    valid = bool(
        metrics.get("auc_valid", False)
        and np.isfinite(metrics["auc"])
        and np.isfinite(metrics["average_precision"])
    )
    auc = (
        metrics["auc"]
        if valid
        else (last_auc if np.isfinite(last_auc) else auc_reference)
    )
    ap = (
        metrics["average_precision"]
        if valid
        else (last_ap if np.isfinite(last_ap) else ap_reference)
    )
    ks, psi = float(metrics["ks"]), float(metrics["psi"])
    if not np.isfinite([ks, psi, auc, ap]).all() or psi < 0:
        raise ValueError("Nonfinite observation or negative PSI")
    result = np.array(
        [ks, psi / (1 + psi), (auc_reference - auc + 1) / 2, ap, float(valid)],
        dtype=np.float32,
    )
    assert result.shape == (5,) and np.isfinite(result).all()
    assert ((result >= -1e-6) & (result <= 1 + 1e-6)).all()
    return result

def select_f1_threshold(labels, probabilities):
    """Maximize positive F1 on A-validation once; choose highest tied threshold."""
    p, r, thresholds = precision_recall_curve(labels, probabilities)
    scores = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-15)
    if len(thresholds) == 0:
        raise ValueError("A-validation cannot identify an F1 threshold")
    return float(thresholds[np.isclose(scores, scores.max(), rtol=0, atol=1e-12)].max())

def classification_metrics(labels, probabilities, threshold=0.5):
    """Compute ranking and fixed-threshold scores with explicit degenerate flags."""
    y, prob = np.asarray(labels, dtype=np.int8), np.asarray(probabilities, dtype=float)
    assert len(y) and len(y) == len(prob) and np.isfinite(prob).all()
    positives, negatives = int((y == 1).sum()), int((y == 0).sum())
    valid = positives > 0 and negatives > 0
    pred = prob >= threshold
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    fn, tn = positives - tp, negatives - fp
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, positives)
    fpos = 2 * tp / max(1, 2 * tp + fp + fn)
    fneg = 2 * tn / max(1, 2 * tn + fp + fn)
    if positives:
        pr, re, _ = precision_recall_curve(y, prob)
        ap, area = float(average_precision_score(y, prob)), float(auc(re, pr))
    else:
        ap = area = np.nan
    return dict(
        auc=float(roc_auc_score(y, prob)) if valid else np.nan,
        average_precision=ap,
        pr_auc_trapz=area,
        ap_valid=positives > 0,
        auc_valid=valid,
        f1_positive=fpos,
        f1_macro=(fpos + fneg) / 2,
        precision=precision,
        recall=recall,
        balanced_accuracy=(recall + tn / negatives) / 2 if valid else np.nan,
        threshold=float(threshold),
        prevalence=positives / len(y),
        positives=positives,
        negatives=negatives,
        rows=len(y),
    )

def read_csv(*args, **kwargs):
    """Parse exported floats without changing threshold-boundary predictions."""
    kwargs.setdefault("float_precision", "round_trip")
    return pd.read_csv(*args, **kwargs)

def atomic_pickle(path, payload):
    """Publish one immutable preparation asset after a complete write."""
    temporary = path.with_suffix(path.suffix + ".pending")
    with gzip.open(temporary, "wb", compresslevel=1) as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary, path)

def load_pickle(path):
    """Read trusted artifacts produced by this private experiment only."""
    with gzip.open(path, "rb") as handle:
        return pickle.load(handle)

## Persistencia por límites confirmados

In [ ]:
def commit_run(stage):
    """Atomically bind complete agent state to exact CSV contents/append offsets."""
    RUN_STATE["stage"] = stage
    sizes, tables = {}, {}
    for path in [*OUT.glob("*.csv"), *(OUT / "predictions").glob("*.csv")]:
        relative = path.relative_to(OUT).as_posix()
        if path.parent.name == "predictions" or path.stem in APPEND_NAMES:
            sizes[relative] = path.stat().st_size
        else:
            tables[relative] = path.read_bytes()
    payload = dict(
        run=RUN_STATE, append_sizes=sizes, tables=tables,
        versions={"torch": torch.__version__, "xgboost": xgboost.__version__,
                  "numpy": np.__version__, "sklearn": sklearn.__version__},
    )
    temporary = OUT / "resume.pending.pt"
    torch.save(payload, temporary)
    os.replace(temporary, OUT / "resume.pt")
    manifest = dict(protocol=RUN_STATE["protocol"], stage=stage, segment=SEGMENT_VERSION,
                    completed_configs=RUN_STATE.get("completed_configs", 0),
                    active_episode=RUN_STATE.get("training", {}).get("episode", 0),
                    requested_configs=1, requested_episodes_per_config=MAX_EPISODES,
                    snapshot_bytes=(OUT / "resume.pt").stat().st_size)
    temporary = OUT / "resume_manifest.pending.json"
    temporary.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    os.replace(temporary, OUT / "resume_manifest.json")

def save_training_state(agent, history, checkpoints, episode, elapsed):
    """Retain training weights separately from selected inference checkpoints."""
    RUN_STATE["training"] = dict(
        config_id=progress["active_config"], episode=episode,
        online=agent.online.state_dict(), target=agent.target.state_dict(),
        optimizer=agent.optim.state_dict(), replay=agent.replay.buffer,
        replay_rng=agent.replay.rng.getstate(), agent_rng=agent.rng.bit_generator.state,
        python_rng=random.getstate(), numpy_rng=np.random.get_state(),
        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(),
        epsilon=agent.epsilon, updates=agent.updates, losses=agent.losses,
        history=history, checkpoints=checkpoints, elapsed_s=elapsed,
    )
    commit_run("training")

def load_training_state(agent, config):
    """Restore every generator and optimizer state before the next full episode."""
    saved = RUN_STATE.get("training")
    if saved is None:
        return [], [], 0, 0.0
    assert saved["config_id"] == config.config_id
    agent.online.load_state_dict(saved["online"])
    agent.target.load_state_dict(saved["target"])
    agent.optim.load_state_dict(saved["optimizer"])
    agent.replay.buffer = saved["replay"]
    agent.replay.rng.setstate(saved["replay_rng"])
    agent.rng.bit_generator.state = saved["agent_rng"]
    random.setstate(saved["python_rng"])
    np.random.set_state(saved["numpy_rng"])
    torch.set_rng_state(saved["torch_rng"])
    torch.cuda.set_rng_state_all(saved["cuda_rng"])
    agent.epsilon, agent.updates, agent.losses = saved["epsilon"], saved["updates"], saved["losses"]
    assert len(saved["history"]) == saved["episode"]
    return saved["history"], saved["checkpoints"], saved["episode"], saved["elapsed_s"]

def initialize_run():
    """Restore the last durable boundary in place; reject changed code, data or mode."""
    global RUN_STATE
    path = OUT/'resume.pt'
    sources = [p.parent for p in Path('/kaggle/input').rglob('resume_manifest.json')] if Path('/kaggle/input').exists() else []
    if len(sources) > 1:
        raise ValueError('Mount only the immediately preceding segment')
    if sources:
        source = sources[0]
        summary = source / 'run_summary.json'
        if not summary.exists() or json.loads(summary.read_text(encoding='utf-8')).get('status') != 'partial':
            raise ValueError('Predecessor must finish with status partial; diagnose failed runs')
        shutil.copytree(source, OUT, dirs_exist_ok=True)
    if path.exists():
        payload = torch.load(path,map_location='cpu',weights_only=False)
        if payload['run']['identity'] != run_identity():
            raise ValueError('Resume identity differs (code, dataset, dependencies or mode)')
        RUN_STATE = payload['run']
        retained = set(payload['append_sizes']) | set(payload['tables'])
        for csv in [*OUT.glob('*.csv'),*(OUT/'predictions').glob('*.csv')]:
            rel = csv.relative_to(OUT).as_posix()
            if rel not in retained:
                csv.unlink()
            elif rel in payload['append_sizes']:
                size = payload['append_sizes'][rel]
                if csv.stat().st_size < size:
                    raise ValueError(f'Truncated durable artifact: {csv}')
                with csv.open('r+b') as handle:
                    handle.truncate(size)
        for rel,size in payload['append_sizes'].items():
            if not (OUT/rel).exists():
                raise FileNotFoundError(rel)
        for rel,content in payload['tables'].items():
            (OUT/rel).write_bytes(content)
    elif list(OUT.glob('*.csv')):
        raise FileExistsError('Outputs exist without a durable snapshot; use a fresh directory')
    else:
        RUN_STATE['identity'] = run_identity()
    reset_seed()

def file_hash(path):
    """Hash a file without loading it into memory."""
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def run_identity():
    """Bind continuations to the notebook, dataset and runtime versions."""
    return dict(protocol=RUN_STATE['protocol'],mode=MODE,episodes=MAX_EPISODES,
        notebook=CODE_ID,
        parquet=file_hash(find_parquet()),torch=torch.__version__,
        numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__,
        xgboost=xgboost.__version__)

def budget_boundary(stage):
    """Stop at a committed boundary, leaving a two-hour margin before the wall limit."""
    if STOP_REQUESTED or time.perf_counter()-RUN_STARTED >= RUN_BUDGET_SECONDS:
        raise SegmentYield(stage)


## Pacientes mixtos y replays causales

In [ ]:
def find_parquet():
    """Locate the explicitly authorized PhysioNet Parquet."""
    explicit = os.environ.get('DARL_DUMMY_PARQUET')
    paths = [Path(explicit)] if explicit else sorted(Path('/kaggle/input').rglob('physionet.parquet'))
    if len(paths) != 1 or not paths[0].is_file():
        raise FileNotFoundError(f'Expected one mounted physionet.parquet, found {len(paths)}')
    return paths[0]

def load_frame():
    """Mix sites at patient level while keeping each hourly history chronological."""
    frame = pd.read_parquet(find_parquet(),columns=['patient_id','source_set','ICULOS','SepsisLabel',*VITALS])
    assert set(frame.source_set.unique()) == {'a','b'}
    assert not frame.duplicated(['patient_id','ICULOS']).any()
    assert frame.groupby('patient_id').source_set.nunique().eq(1).all()
    frame = frame.sort_values(['patient_id','ICULOS']).reset_index(drop=True)
    frame[VITALS] = frame.groupby('patient_id',sort=False)[VITALS].ffill()
    patients = frame.groupby(['source_set','patient_id']).SepsisLabel.max().reset_index()
    strata = patients.source_set.astype(str)+'_'+patients.SepsisLabel.astype(str)
    train,rest = train_test_split(patients.index,test_size=.4,stratify=strata,random_state=SEED)
    val,test = train_test_split(rest,test_size=.5,stratify=strata.loc[rest],random_state=SEED)
    parts = {name:sorted(patients.loc[ix,'patient_id']) for name,ix in
             [('train',train),('validation',val),('test',test)]}
    assert all(not set(parts[a]) & set(parts[b]) for a,b in itertools.combinations(parts,2))
    manifest = patients.copy()
    manifest['partition'] = manifest.patient_id.map({pid:k for k,ids in parts.items() for pid in ids})
    table('patient_partitions',manifest)
    return frame,parts

def episode_manifest(episodes):
    """Freeze replay choice, scenario and post-base drift independent of actions."""
    rng = np.random.default_rng(SEED)
    return pd.DataFrame([dict(episode=e,scenario=TRAIN_CYCLE[(e-1)%len(TRAIN_CYCLE)],
        replay_id=(e-1)%4,onset=int(rng.integers(8,13)),
        rate_factor=float(rng.choice((.75,1.,1.25)))) for e in range(1,episodes+1)])


def make_replay(frame, ids, days=DAYS, beds=BEDS, rng=None):
    """Consume each patient's hourly sequence once; refill a bed next hour."""
    rng = np.random.default_rng(SEED) if rng is None else rng
    assert days >= 1 and beds > 0 and len(set(ids)) == len(ids)
    histories = {
        pid: g.index.to_numpy()
        for pid, g in frame.loc[frame.patient_id.isin(ids)].groupby(
            "patient_id", sort=False
        )
    }
    queue = list(rng.permutation(sorted(ids)))
    if len(queue) < beds:
        raise RuntimeError(
            f"Patient queue exhausted before initialization: {len(queue)} patients, {beds} beds"
        )
    slots = [(queue.pop(), 0) for _ in range(beds)]
    indices = []
    for hour in range(days * 24):
        for bed, (pid, pos) in enumerate(slots):
            indices.append(int(histories[pid][pos]))
            pos += 1
            if pos == len(histories[pid]) and hour < days * 24 - 1:
                if not queue:
                    raise RuntimeError(
                        f"Patient queue exhausted at hour {hour}; requested {days} days, {beds} beds, pool={len(ids)}"
                    )
                slots[bed] = (queue.pop(), 0)
            else:
                slots[bed] = (pid, pos)
    out = frame.loc[indices].reset_index(drop=True).copy()
    out["hour"] = np.repeat(np.arange(days * 24), beds)
    out["bed"] = np.tile(np.arange(beds), days * 24)
    out["day"] = out.hour // 24 + 1
    out["label_available_day"] = out.day + LABEL_DELAY
    assert len(out) == days * 24 * beds and out.groupby("hour").size().eq(beds).all()
    assert not out.duplicated(["patient_id", "ICULOS"]).any()
    assert (
        out.groupby("patient_id")
        .ICULOS.apply(lambda x: x.is_monotonic_increasing)
        .all()
    )
    return out

def build_patient_metadata(frame, partitions=None):
    """Index COMPLETE histories; simulation metadata must never be model inputs."""
    membership = {pid: name for name, ids in (partitions or {}).items() for pid in ids}
    metadata = {}
    for (site, pid), rows in frame.groupby(["source_set", "patient_id"], sort=False):
        hours = np.sort(rows.ICULOS.to_numpy(dtype=float))
        if len(np.unique(hours)) != len(hours) or not np.isfinite(hours).all():
            raise ValueError(f"Invalid patient history: {site}/{pid}")
        positive = rows.loc[rows.SepsisLabel.eq(1), "ICULOS"]
        metadata[(site, pid)] = dict(
            source_set=site, patient_id=pid, partition=membership.get(pid, str(site)),
            hours=hours, first_iculos=float(hours[0]), last_iculos=float(hours[-1]),
            original_positive=bool(len(positive)),
            original_first_positive=float(positive.min()) if len(positive) else None,
        )
    return metadata

def concept_uniform(config, metadata, admission_hour, purpose):
    """Stable independent draws, shared by policies, reward weights and scenarios."""
    key = json.dumps([str(config.seed), str(metadata["partition"]),
                      str(metadata["source_set"]), str(metadata["patient_id"]),
                      str(int(admission_hour)), purpose], separators=(",", ":"))
    # 53 bits guarantee a double in [0, 1), including at the largest hash value.
    bits = int.from_bytes(hashlib.sha256(key.encode("utf-8")).digest()[:8], "big") >> 11
    return bits / 2**53

def concept_audit_frame(schedule):
    """Recover patient decisions from immutable simulation-only DataFrame attrs."""
    return pd.DataFrame(json.loads(schedule.attrs.get("concept_audit_json", "[]")))

def inject(schedule, scenario, cov_rate, concept_rate, onset, med, iqr,
           patient_metadata=None, concept_config=None):
    """Apply IQR feature drift and one frozen synthetic onset per new admission."""
    if scenario not in SCENARIOS or cov_rate < 0 or concept_rate < 0:
        raise ValueError("Unknown scenario or negative drift rate")
    config = TemporalConceptConfig() if concept_config is None else concept_config
    conceptual = scenario in ("concept", "combined")
    if conceptual and patient_metadata is None:
        raise ValueError("Full-history patient_metadata is required for concept drift")
    out = schedule.copy()
    days = out.day.to_numpy()
    cov = (np.maximum(0, days - onset + 1) * cov_rate
           if scenario in ("covariate", "combined") else np.zeros(len(out)))
    concept_start = int(onset + (CONCEPT_DELAY if scenario == "combined" else 0))
    concept_level = (np.minimum(CONCEPT_MAX, np.maximum(0, days-concept_start+1)*concept_rate)
                     if conceptual else np.zeros(len(out)))
    raw = out[VITALS].to_numpy(dtype=np.float32)
    med_array, iqr_array = np.asarray(med, dtype=np.float32), np.asarray(iqr, dtype=np.float32)
    scale = 1 + SCALE_GAIN * np.minimum(1, cov / 3)
    direction = np.array([1, -1, 1, 1, -1, 1, 1], dtype=np.float32)
    if scenario in ("covariate", "combined"):
        shifted = med_array + scale[:, None]*(raw-med_array) + cov[:, None]*iqr_array*direction
        out[VITALS] = np.where(cov[:, None] > 0, shifted, raw)

    labels = out.SepsisLabel.to_numpy(dtype=np.int8).copy()
    hours = out.ICULOS.to_numpy(dtype=float)
    global_hours = out.hour.to_numpy()
    audit = []
    if conceptual:
        for key, positions in out.groupby(["source_set", "patient_id"], sort=False).indices.items():
            if key not in patient_metadata:
                raise ValueError(f"Missing full history for {key}")
            info = patient_metadata[key]
            admission_hour = int(global_hours[positions].min())
            admission_day = admission_hour // 24 + 1
            intensity = float(np.clip((admission_day-concept_start+1)*concept_rate/CONCEPT_MAX, 0, 1))
            eligible = admission_day >= concept_start and intensity > 0
            maximum = int(np.floor(config.max_shift_hours*intensity))
            probability = config.negative_event_probability*intensity
            original = info["original_first_positive"]
            synthetic = original
            converted = False
            draw = concept_uniform(config, info, admission_hour, "convert")
            shift_draw = concept_uniform(config, info, admission_hour, "shift")
            onset_draw = concept_uniform(config, info, admission_hour, "onset")
            reason = "admitted_before_drift" if admission_day < concept_start else "zero_intensity"
            if eligible:
                if info["original_positive"]:
                    shift = info["hours"] - original
                    allowed = info["hours"][(shift >= -maximum) & (shift <= 0)]
                    synthetic = float(allowed[int(shift_draw*len(allowed))])
                    reason = "positive_shifted" if synthetic != original else "positive_unchanged"
                elif len(info["hours"]) < 2:
                    reason = "insufficient_history"
                elif draw < probability:
                    allowed = info["hours"][1:]
                    synthetic = float(allowed[int(onset_draw*len(allowed))])
                    converted, reason = True, "negative_converted"
                else:
                    reason = "negative_unchanged"
                labels[positions] = (hours[positions] >= synthetic).astype(np.int8) if synthetic is not None else 0
                ordered = positions[np.argsort(hours[positions])]
                assert np.all(np.diff(labels[ordered]) >= 0)
            visible = bool(np.any(labels[positions] == 1))
            audit.append(dict(
                injector_version=config.version, scenario=scenario, source_set=str(key[0]),
                patient_id=str(key[1]), partition=info["partition"],
                admission_hour=admission_hour, admission_day=admission_day,
                concept_start_day=concept_start, concept_rate=float(concept_rate), intensity=intensity,
                eligible=eligible, max_shift_hours=maximum,
                original_positive=info["original_positive"],
                original_first_positive=original, synthetic_first_positive=synthetic,
                first_iculos=info["first_iculos"], last_iculos=info["last_iculos"],
                effective_shift_hours=synthetic-original if original is not None else None,
                conversion_probability=probability if eligible and not info["original_positive"] else 0.,
                conversion_draw=draw, shift_draw=shift_draw, onset_draw=onset_draw,
                converted=converted, reason=reason, event_visible_in_replay=visible,
                event_censored_in_replay=synthetic is not None and not visible,
                replay_last_iculos=float(hours[positions].max()), replay_rows=len(positions),
            ))
    out["scenario_label"], out["scenario"] = labels, scenario
    out["cov_level"], out["concept_level"] = cov, concept_level
    out.attrs["injector_version"] = config.version
    out.attrs["concept_audit_json"] = json.dumps(audit)
    out.attrs["label_schedule_id"] = hashlib.sha256(
        out.attrs["concept_audit_json"].encode("utf-8")).hexdigest()[:20] if audit else "original_labels"
    assert out.SepsisLabel.equals(schedule.SepsisLabel)
    assert out.scenario_label.isin([0, 1]).all()
    if not conceptual:
        assert np.array_equal(out.scenario_label, out.SepsisLabel.to_numpy())
    return out

def persist_concept_audit(schedule, phase, replay_id):
    """Export each shared label schedule once, independent of reward configuration."""
    audit = concept_audit_frame(schedule)
    if audit.empty:
        return
    payload = schedule.attrs["concept_audit_json"]
    identifier = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]
    directory = OUT / "concept_audits"
    directory.mkdir(exist_ok=True)
    path = directory / f"{phase}_{replay_id}_{schedule.scenario.iloc[0]}_{identifier}.csv"
    if not path.exists():
        audit.insert(0, "schedule_id", identifier)
        audit.insert(1, "phase", phase)
        audit.insert(2, "replay_id", replay_id)
        audit.to_csv(path, index=False)

@dataclass(frozen=True)
class TemporalConceptConfig:
    """Fixed limits for the semisynthetic patient-onset perturbation."""

    seed: int = 42
    max_shift_hours: int = 24
    negative_event_probability: float = 0.10
    version: str = "temporal_advance_onset_v2"

    def __post_init__(self):
        """Reject limits outside the agreed protocol, including noninteger hours."""
        if not 0 <= self.max_shift_hours <= 24 or int(self.max_shift_hours) != self.max_shift_hours:
            raise ValueError("max_shift_hours must be an integer in [0, 24]")
        if not 0 <= self.negative_event_probability <= .10:
            raise ValueError("negative_event_probability must be in [0, 0.10]")

TEMPORAL_CONCEPT_CONFIG = TemporalConceptConfig()

## Pipeline de dos etapas de v7

In [ ]:
def build_reference(rows):
    """Freeze raw per-vital deciles and a bounded KS sample."""
    result = {}
    for col in VITALS:
        clean = rows[col].dropna().to_numpy(dtype=float)
        clean = clean[np.isfinite(clean)]
        if not len(clean):
            result[col] = None
            continue
        edges = np.unique(np.quantile(clean, np.linspace(0, 1, 11)))
        edges = (
            np.r_[-np.inf, edges[1:-1], np.inf]
            if len(edges) > 2
            else np.array([-np.inf, np.inf])
        )
        result[col] = dict(
            sample=clean[:: max(1, len(clean) // 1000)][:1000],
            edges=edges,
            hist=np.histogram(clean, bins=edges)[0],
        )
    return result

class Pipeline:
    """Robust preprocessing and XGBoost with independently replaceable stages."""

    def __init__(self, med, iqr, model, ref, params, version=0):
        self.med = np.asarray(med, dtype=np.float32).copy()
        self.iqr = np.asarray(iqr, dtype=np.float32).copy()
        self.model, self.ref, self.params, self.version = (
            model,
            ref,
            dict(params),
            version,
        )

    def transform(self, rows):
        """Impute with past medians, robust-scale and clip values."""
        raw = rows[VITALS].to_numpy(dtype=np.float32)
        return np.clip(
            (np.where(np.isfinite(raw), raw, self.med) - self.med) / self.iqr, -2, 2
        )

    def predict(self, rows):
        """Return probabilities without consulting any labels."""
        return self.model.predict_proba(self.transform(rows))[:, 1].astype(np.float64)

    def saturation(self, rows):
        """Return the share of feature cells clipped by the current transformer."""
        raw = rows[VITALS].to_numpy(dtype=np.float32)
        return float(
            np.mean(
                np.abs(
                    (np.where(np.isfinite(raw), raw, self.med) - self.med) / self.iqr
                )
                > 2
            )
        )

    def act(self, action, rows):
        """Build a new pipeline using exclusively the supplied labeled past window."""
        assert 0 <= action < 4
        start = time.perf_counter()
        if action == 0:
            return self, time.perf_counter() - start
        med, iqr = self.med, self.iqr
        if action in (1, 3):
            med = (
                rows[VITALS]
                .median()
                .fillna(pd.Series(self.med, index=VITALS))
                .to_numpy(dtype=np.float32)
            )
            q = rows[VITALS].quantile(0.75) - rows[VITALS].quantile(0.25)
            iqr = (
                q.fillna(pd.Series(self.iqr, index=VITALS))
                .clip(lower=1e-6)
                .to_numpy(dtype=np.float32)
            )
        model = self.model
        if action in (2, 3):
            y = rows.scenario_label.to_numpy(dtype=np.int8)
            if np.unique(y).size != 2:
                raise ValueError("Retraining requires both classes in the past window")
            model = XGBClassifier(
                **self.params,
                tree_method="hist",
                device="cpu",
                n_jobs=2,
                random_state=SEED,
                eval_metric="logloss",
                scale_pos_weight=float((y == 0).sum() / max(1, (y == 1).sum())),
            )
            model.fit(
                Pipeline(med, iqr, None, self.ref, self.params).transform(rows), y
            )
        ref = build_reference(rows) if action in (1, 3) else self.ref
        return (
            Pipeline(med, iqr, model, ref, self.params, self.version + 1),
            time.perf_counter() - start,
        )

def fit_pipeline(rows, params):
    """Fit both initial stages from an A-train window only."""
    med = rows[VITALS].median().fillna(0.0).to_numpy(dtype=np.float32)
    iqr = (
        (rows[VITALS].quantile(0.75) - rows[VITALS].quantile(0.25))
        .fillna(1.0)
        .clip(lower=1e-6)
        .to_numpy(dtype=np.float32)
    )
    base = Pipeline(med, iqr, None, build_reference(rows), params)
    fitted, _ = base.act(2, rows)
    fitted.version = 0
    return fitted

def monitor_features(rows, pipe):
    """Compute KS/PSI on exactly the daily rows receiving predictions."""
    ks = []
    psis = []
    for col in VITALS:
        current = rows[col].dropna().to_numpy(dtype=float)
        current = current[np.isfinite(current)]
        ref = pipe.ref[col]
        if not len(current) or ref is None:
            continue
        ks.append(float(ks_2samp(ref["sample"], current, method="asymp").statistic))
        p = ref["hist"].astype(float) + 1e-6
        q = np.histogram(current, bins=ref["edges"])[0].astype(float) + 1e-6
        p /= p.sum()
        q /= q.sum()
        psis.append(float(np.sum((q - p) * np.log(q / p))))
    return dict(
        ks=max(ks, default=0.0),
        psi=max(psis, default=0.0),
        monitored_vitals=len(ks),
        saturation=pipe.saturation(rows),
    )

## Double DQN heredado de v7

In [ ]:
class QNetwork(nn.Module):
    """Four action values from the last three daily observation blocks."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(OBS_DIM, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
        )

    def forward(self, x):
        """Return unconstrained Q values."""
        return self.net(x)

class ReplayBuffer:
    """FIFO transitions including next-state feasible-action masks."""

    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
        self.rng = random.Random(SEED)

    def push(self, s, a, r, s2, done, mask):
        """Store immutable transition snapshots."""
        self.buffer.append(
            (s.copy(), int(a), float(r), s2.copy(), bool(done), mask.copy())
        )

    def sample(self, n):
        """Sample uniformly from prior causal decisions."""
        return tuple(np.stack(x) for x in zip(*self.rng.sample(self.buffer, n)))

    def __len__(self):
        """Return the retained transition count."""
        return len(self.buffer)

class DQNAgent:
    """Epsilon-greedy Double DQN; only training episodes update its weights."""

    def __init__(self):
        self.online = QNetwork().to(DEVICE)
        self.target = QNetwork().to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.target.requires_grad_(False)
        self.optim = torch.optim.Adam(self.online.parameters(), lr=LR_START)
        self.replay = ReplayBuffer()
        self.rng = np.random.default_rng(SEED)
        self.epsilon = 1.0
        self.updates = 0
        self.losses = []

    def q_values(self, s):
        """Return online action values without constructing gradients."""
        with torch.no_grad():
            x = torch.as_tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            return self.online(x).squeeze(0).cpu().numpy()

    def select(self, s, mask, greedy=False):
        """Explore or maximize over available maintenance actions."""
        allowed = np.flatnonzero(mask)
        if not greedy and self.rng.random() < self.epsilon:
            return int(self.rng.choice(allowed))
        return int(np.argmax(np.where(mask, self.q_values(s), -np.inf)))

    def learn(self, s, a, r, s2, done, next_mask):
        """Take one masked Double DQN gradient step after a label release."""
        assert np.isfinite(r) and next_mask[0]
        self.replay.push(s, a, r, s2, done, next_mask)
        if len(self.replay) < 64:
            return None
        states, actions, rewards, following, terminals, masks = self.replay.sample(64)
        states = torch.as_tensor(states, dtype=torch.float32, device=DEVICE)
        actions = torch.as_tensor(actions, dtype=torch.long, device=DEVICE).unsqueeze(1)
        following = torch.as_tensor(following, dtype=torch.float32, device=DEVICE)
        rewards = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
        terminals = torch.as_tensor(terminals, dtype=torch.float32, device=DEVICE)
        masks = torch.as_tensor(masks, dtype=torch.bool, device=DEVICE)
        values = self.online(states).gather(1, actions).squeeze(1)
        with torch.no_grad():
            selected = (
                self.online(following)
                .masked_fill(~masks, -torch.inf)
                .argmax(1, keepdim=True)
            )
            next_values = self.target(following).gather(1, selected).squeeze(1)
            targets = rewards + 0.99 * (1 - terminals) * next_values
        loss = nn.functional.smooth_l1_loss(values, targets)
        self.optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(self.online.parameters(), 10.0)
        self.optim.step()
        self.updates += 1
        if self.updates % 50 == 0:
            self.target.load_state_dict(self.online.state_dict())
        self.losses.append(float(loss.item()))
        return self.losses[-1]

    def end_episode(self, fraction):
        """Decay exploration and learning rate once per episode."""
        self.epsilon = max(EPS_MIN, self.epsilon * EPS_DECAY)
        for g in self.optim.param_groups:
            g["lr"] = LR_START + (LR_END - LR_START) * min(1.0, fraction)

    @property
    def lr(self):
        """Return the current learning rate."""
        return self.optim.param_groups[0]["lr"]

    def fingerprint(self):
        """Hash online weights to prove validation/test do not train the agent."""
        h = hashlib.sha256()
        for tensor in self.online.state_dict().values():
            h.update(tensor.detach().cpu().numpy().tobytes())
        return h.hexdigest()

## Entorno pre/post y etiquetas diferidas

In [ ]:
class DailyEnv:
    """Predict pre/post on day t, release labels on t+1, then learn one transition."""
    def __init__(self,schedule,initial,auc_reference,cost_unit,ap_reference=0.,
                 reward_config=COMMON_CONFIG,threshold=.5,prediction_path=None,
                 policy='dqn',phase='train',episode=0):
        self.schedule = schedule
        self.parts = {int(d):r.reset_index(drop=True) for d,r in schedule.groupby('day',sort=True)}
        self.days = int(schedule.day.max())
        assert self.days > WARMUP
        self.pipe,self.auc_reference,self.ap_reference = initial,auc_reference,ap_reference
        self.last_auc,self.last_ap = auc_reference,ap_reference
        self.cost_unit,self.config,self.threshold = cost_unit,reward_config,threshold
        self.prediction_path,self.policy = prediction_path,policy
        self.context = dict(config_id=reward_config.config_id,**asdict(reward_config),
            injector_version=schedule.attrs.get('injector_version',INJECTOR_VERSION),
            label_schedule_id=schedule.attrs.get('label_schedule_id','original_labels'),
            phase=phase,episode=episode)
        neutral = np.array([0.,0.,.5,ap_reference,0.],dtype=np.float32)
        self.history = deque([neutral.copy() for _ in range(3)],maxlen=3)
        self.daily,self.actions = [],[]
        self.day = WARMUP+1

    def state(self):
        """Expose only released observations; bootstrap blocks are explicitly invalid."""
        out = np.concatenate(self.history).astype(np.float32)
        assert out.shape == (OBS_DIM,) and np.isfinite(out).all()
        return out

    def training_rows(self):
        """Use exactly seven preceding days, whose labels have all arrived."""
        out = pd.concat([self.parts[d] for d in range(self.day-UPDATE_DAYS,self.day)],ignore_index=True)
        assert out.day.nunique() == UPDATE_DAYS and out.day.max() < self.day
        assert out.label_available_day.max() <= self.day
        return out

    def action_mask(self):
        """Mask classifier updates when the past window has only one class."""
        both = self.training_rows().scenario_label.nunique() == 2
        return np.array([True,True,both,both],dtype=bool)

    def step(self,requested):
        """Run exactly one action after freezing pre-update probabilities."""
        if self.day > self.days:
            raise RuntimeError('Episode already finished')
        d,state,past,mask = self.day,self.state(),self.training_rows(),self.action_mask()
        executed = int(requested) if mask[int(requested)] else 0
        rows = self.parts[d]
        pre_version = self.pipe.version
        started = time.perf_counter()
        pre_prob = np.array(self.pipe.predict(rows),copy=True)
        pre_prediction_s = time.perf_counter()-started
        self.pipe,seconds = self.pipe.act(executed,past)
        started = time.perf_counter()
        post_prob = pre_prob if executed == 0 else np.array(self.pipe.predict(rows),copy=True)
        post_prediction_s = 0. if executed == 0 else time.perf_counter()-started
        feature_metrics = monitor_features(rows,self.pipe)
        # The prediction arrays remain frozen until the simulated label-release event.
        self.day = d+LABEL_DELAY
        assert rows.label_available_day.le(self.day).all()
        pre = classification_metrics(rows.scenario_label,pre_prob,self.threshold)
        post = classification_metrics(rows.scenario_label,post_prob,self.threshold)
        original = classification_metrics(rows.SepsisLabel,post_prob,self.threshold)
        components = reward_components(post,pre,seconds,self.cost_unit,self.config)
        record = dict(**self.context,day=d,policy=self.policy,scenario=rows.scenario.iloc[0],
            **feature_metrics,**post,**{f'{k}_pre':v for k,v in pre.items()},
            **{f'{k}_post':v for k,v in post.items()},
            **{f'{k}_original':v for k,v in original.items()},**components,
            action=ACTIONS[executed],requested=int(requested),executed=executed,
            time_s=seconds,cost_unit_s=self.cost_unit,pre_prediction_s=pre_prediction_s,
            post_prediction_s=post_prediction_s,pipeline_version=self.pipe.version,
            pipeline_version_pre=pre_version,label_received_day=self.day,warmup=False,
            auc_drop=self.auc_reference-post['auc'],patients=int(rows.patient_id.nunique()),
            cov_level=float(rows.cov_level.iloc[0]),concept_level=float(rows.concept_level.iloc[0]),
            state_end_day=d-1,training_min_day=int(past.day.min()),training_max_day=int(past.day.max()),
            training_label_available_max=int(past.label_available_day.max()),
            fallback_reason='' if requested == executed else 'one_class_past_window',
            **{f'mask_{a}':bool(mask[i]) for i,a in enumerate(ACTIONS)},
            **{f'state_{i}':float(v) for i,v in enumerate(state)})
        block = encode_observation(record,self.last_auc,self.last_ap,self.auc_reference,self.ap_reference)
        record.update({f'obs_{name}':float(block[i]) for i,name in enumerate(OBS_NAMES)})
        self.history.append(block)
        if components['predictive_valid']:
            self.last_auc,self.last_ap = post['auc'],post['average_precision']
        self.daily.append(record)
        self.actions.append(record)
        if self.prediction_path is not None:
            columns = ['patient_id','source_set','ICULOS','hour','day','bed','SepsisLabel',
                       'scenario_label','label_available_day']
            output = rows[columns].copy()
            output['probability_pre'],output['probability_post'] = pre_prob,post_prob
            output['probability'] = post_prob
            for key in ('config_id','policy','scenario','phase','episode','pipeline_version',
                        'pipeline_version_pre','action','label_received_day'):
                output[key] = record[key]
            path = Path(self.prediction_path)
            output.to_csv(path,mode='a',header=not path.exists(),index=False,float_format='%.17g')
        return self.state(),record['reward'],d == self.days,record


## prepare

In [ ]:
def prepare_initial(frame, ids):
    """Select eight XGBoost candidates on a train-only patient holdout, first 7 days."""
    patients = frame.loc[frame.patient_id.isin(ids)].groupby('patient_id').agg(
        source_set=('source_set','first'),positive=('SepsisLabel','max'))
    strata = patients.source_set.astype(str)+'_'+patients.positive.astype(str)
    fit_ids,holdout_ids = train_test_split(patients.index,test_size=.2,stratify=strata,random_state=SEED)
    fit = make_replay(frame,fit_ids,days=7)
    val = make_replay(frame,holdout_ids,days=7)
    fit['scenario_label'],val['scenario_label'] = fit.SepsisLabel,val.SepsisLabel
    assert fit.scenario_label.nunique() == val.scenario_label.nunique() == 2
    table('initial_holdout_patients',[dict(patient_id=p,role=k) for k,group in
        [('fit',fit_ids),('holdout',holdout_ids)] for p in group])
    candidates,best,rank = [],None,None
    for i,(depth,lr,trees) in enumerate(itertools.product((3,5),(.05,.1),(60,120))):
        params = dict(max_depth=depth,learning_rate=lr,n_estimators=trees)
        pipe = fit_pipeline(fit,params)
        probabilities = pipe.predict(val)
        metrics = classification_metrics(val.scenario_label,probabilities)
        candidate = (metrics['auc'],metrics['average_precision'],-trees,-depth,-lr)
        candidates.append(dict(candidate=i,**params,**metrics))
        if rank is None or candidate > rank:
            best,rank = (params,probabilities,pipe),candidate
    table('xgboost_search',candidates)
    params,probabilities,pipe = best
    threshold = select_f1_threshold(val.scenario_label,probabilities)
    metrics = classification_metrics(val.scenario_label,probabilities,threshold)
    table('f1_threshold',[dict(threshold=threshold,source='train_internal_patient_holdout_days_1_7')])
    table('observation_reference',[dict(auc_reference=metrics['auc'],ap_reference=metrics['average_precision'],
        source='train_internal_patient_holdout_days_1_7',history_days=3,dimension=OBS_DIM)])
    table('initial_config',[dict(**params,**metrics,source='train_only_first_seven_days')])
    return params,metrics['auc'],metrics['average_precision'],threshold


def calibrate(base, initial, auc_reference, patient_metadata):
    """Calibrate drift and fixed seven-day A4 time reference on train only."""
    history = base.loc[base.day.le(WARMUP)].copy()
    history['scenario_label'] = history.SepsisLabel
    assert history.scenario_label.nunique() == 2
    timings = [initial.act(3,history)[1] for _ in range(5)]
    unit = max(float(np.median(timings)),1e-6)
    table('cost_calibration',[dict(repetition=i+1,a4_seconds=v,cost_unit=unit,
        days=7,rows=len(history),source='train') for i,v in enumerate(timings)])
    candidates,rates = [],{}
    stable = inject(base,'stable',0,0,ONSET,initial.med,initial.iqr)
    for kind,grid,repair in [('covariate',COV_RATE_GRID,1),('concept',CONCEPT_RATE_GRID,2)]:
        for rate in grid:
            schedule = inject(base,kind,rate if kind=='covariate' else 0.,
                rate if kind=='concept' else 0.,ONSET,initial.med,initial.iqr,patient_metadata,TEMPORAL_CONCEPT_CONFIG)
            past = schedule.loc[schedule.day.between(DAYS-UPDATE_DAYS,DAYS-1)]
            changed,_ = initial.act(repair,past) if repair==1 or past.scenario_label.nunique()==2 else (initial,0.)
            def score(rows,pipe):
                """Score simulator outcomes strictly inside the training partition."""
                return classification_metrics(rows.scenario_label,pipe.predict(rows))['auc']
            early0 = score(stable.loc[stable.day.eq(ONSET)],initial)
            late0 = score(stable.loc[stable.day.eq(DAYS)],initial)
            early = score(schedule.loc[schedule.day.eq(ONSET)],initial)
            late = score(schedule.loc[schedule.day.eq(DAYS)],initial)
            recovery = score(schedule.loc[schedule.day.eq(DAYS)],changed)-late
            qualified = bool(np.isfinite([early0,late0,early,late,recovery]).all()
                and early0-early <= .05 and late0-late >= .1 and recovery >= .05)
            candidates.append(dict(kind=kind,rate=rate,early_drop=early0-early,
                late_drop=late0-late,recovery=recovery,qualified=qualified))
        subset = pd.DataFrame([r for r in candidates if r['kind']==kind])
        qualified = subset.loc[subset.qualified]
        chosen = (qualified.sort_values('rate') if len(qualified) else subset.sort_values(
            ['late_drop','rate'],ascending=[False,True],na_position='last')).iloc[0]
        rates[kind] = float(chosen.rate)
    report = table('calibration',candidates)
    return rates['covariate'],rates['concept'],unit,bool(report.groupby('kind').qualified.any().all())


def prepare_shared():
    """Persist replay-specific seven-day initial models, training calibration and schedules."""
    cache = OUT/'prepared.pkl.gz'
    if RUN_STATE.get('prepared'):
        return load_pickle(cache)
    frame,parts = load_frame()
    metadata = build_patient_metadata(frame,parts)
    params,auc_ref,ap_ref,threshold = prepare_initial(frame,parts['train'])
    rng = np.random.default_rng(SEED)
    bases = {f'train_{i}':make_replay(frame,parts['train'],rng=rng) for i in range(4)}
    bases.update({k:make_replay(frame,parts[k]) for k in ('validation','test')})
    initials,counts,transforms = {},[],[]
    for name,replay in bases.items():
        history = replay.loc[replay.day.le(WARMUP)].copy()
        history['scenario_label'] = history.SepsisLabel
        if history.scenario_label.nunique()!=2:
            raise ValueError(f'Initial seven-day window lacks both classes: {name}')
        initials[name] = fit_pipeline(history,params)
        counts.append(dict(replay=name,patients_used=replay.patient_id.nunique(),
            rows=len(replay),days=replay.day.max(),beds=BEDS,capacity_passed=True,
            initial_min_day=history.day.min(),initial_max_day=history.day.max()))
        transforms.extend(dict(replay=name,vital=v,median=float(initials[name].med[i]),
            iqr=float(initials[name].iqr[i])) for i,v in enumerate(VITALS))
    table('partition_counts',counts)
    table('initial_transforms',transforms)
    cov,concept,unit,qualified = calibrate(bases['train_0'],initials['train_0'],auc_ref,metadata)
    assets = dict(bases=bases,initials=initials,metadata=metadata,auc_reference=auc_ref,
        ap_reference=ap_ref,threshold=threshold,cov_rate=cov,concept_rate=concept,
        cost_unit=unit,calibration_qualified=qualified,manifest=episode_manifest(MAX_EPISODES))
    table('episode_manifest',assets['manifest'])
    atomic_pickle(cache,assets)
    RUN_STATE['prepared'] = True
    commit_run('prepared')
    budget_boundary('prepared')
    return assets


def scenario_schedule(assets,name,scenario,onset=ONSET,rate_factor=1.):
    """Inject drift relative to this replay's own untouched initial pipeline."""
    initial = assets['initials'][name]
    schedule = inject(assets['bases'][name],scenario,assets['cov_rate']*rate_factor,
        assets['concept_rate']*rate_factor,onset,initial.med,initial.iqr,
        assets['metadata'],TEMPORAL_CONCEPT_CONFIG)
    assert np.array_equal(
        schedule.loc[schedule.day.le(WARMUP), 'scenario_label'].to_numpy(),
        assets['bases'][name].loc[assets['bases'][name].day.le(WARMUP), 'SepsisLabel'].to_numpy())
    return schedule


def env_arguments(assets,name):
    """Bind the replay-specific initial pipeline to frozen train-only references."""
    return dict(initial=assets['initials'][name],auc_reference=assets['auc_reference'],
        ap_reference=assets['ap_reference'],cost_unit=assets['cost_unit'],
        threshold=assets['threshold'],reward_config=COMMON_CONFIG)


def export_drift_tables(schedules):
    """Export raw drift samples, patient decisions and small temporal plot tables."""
    samples, calendar, audits, timelines = [], [], [], []
    for scenario, schedule in schedules.items():
        audit = concept_audit_frame(schedule)
        if not audit.empty:
            audit["phase"], audit["replay_id"] = "test", "mixed_test"
            audits.append(audit)
            persist_concept_audit(schedule, "test", "mixed_test")
            categories = [audit.reason.eq("admitted_before_drift"),
                          audit.effective_shift_hours.lt(0), audit.original_positive & audit.effective_shift_hours.eq(0),
                          audit.converted, audit.reason.eq("negative_unchanged")]
            selected = []
            for condition in categories:
                subset = audit.loc[condition].sort_values(
                    ["event_censored_in_replay", "replay_rows", "source_set", "patient_id"],
                    ascending=[True, False, True, True])
                if not subset.empty:
                    row = subset.iloc[0]
                    selected.append((row.source_set, row.patient_id))
            if not selected:
                selected = list(audit[["source_set", "patient_id"]].head(5).itertuples(index=False, name=None))
            for site, pid in dict.fromkeys(selected):
                mask = schedule.source_set.astype(str).eq(str(site)) & schedule.patient_id.astype(str).eq(str(pid))
                rows = schedule.loc[mask, ["source_set", "patient_id", "ICULOS", "hour", "day",
                                           "scenario", "SepsisLabel", "scenario_label"]].copy()
                info = audit.loc[audit.source_set.eq(site) & audit.patient_id.eq(pid)].iloc[0]
                for key in ("reason", "admission_day", "concept_start_day", "original_first_positive",
                            "synthetic_first_positive", "effective_shift_hours", "event_censored_in_replay"):
                    rows[key] = info[key]
                timelines.append(rows)
        for day, g in schedule.groupby("day"):
            admitted = audit.loc[audit.admission_day.eq(day)] if not audit.empty else audit
            calendar.append(dict(
                injector_version=schedule.attrs["injector_version"], scenario=scenario, day=day,
                cov_level=float(g.cov_level.iloc[0]), concept_level=float(g.concept_level.iloc[0]),
                original_prevalence=float(g.SepsisLabel.mean()), scenario_prevalence=float(g.scenario_label.mean()),
                flip_rate=float(np.mean(g.scenario_label != g.SepsisLabel)),
                patients=int(g.patient_id.nunique()), rows=len(g),
                concept_start_day=int(audit.concept_start_day.iloc[0]) if not audit.empty else np.nan,
                new_admissions=len(admitted),
                eligible_admissions=int(admitted.eligible.sum()) if len(admitted) else 0,
                converted_admissions=int(admitted.converted.sum()) if len(admitted) else 0,
                positive_admissions=int(admitted.original_positive.sum()) if len(admitted) else 0,
            ))
        if scenario in ("covariate", "concept"):
            rows = schedule.loc[schedule.day.isin([1, int(schedule.day.max())])]
            samples.append(rows[["scenario", "day", "SepsisLabel", "scenario_label", *VITALS]].melt(
                id_vars=["scenario", "day", "SepsisLabel", "scenario_label"],
                value_vars=VITALS, var_name="vital", value_name="value"))
    table("drift_calendar", calendar)
    table("drift_samples", pd.concat(samples, ignore_index=True))
    if audits:
        table("concept_patients", pd.concat(audits, ignore_index=True))
    if timelines:
        table("concept_timelines", pd.concat(timelines, ignore_index=True))

## training

In [ ]:
def summarize_roll(daily,actions,threshold,probabilities=None,labels=None):
    """Summarize only decision days, keeping temporal means and pooled scores distinct."""
    first = actions.iloc[0]
    result = {key:first[key] for key in ('config_id','alpha','beta','eta_auc','eta_ap',
        'cost_lambda','phase','episode','scenario','policy','label_schedule_id')}
    result.update(native_return=float(actions.reward.sum()),common_return=float(actions.reward.sum()),
        cost_relative=float(actions.cost_relative.sum()),cost_penalty=float(actions.cost_penalty.sum()),
        seconds=float(actions.time_s.sum()),decisions=len(actions),days=DAYS,
        decision_days=len(daily),mean_daily_ap=float(daily.average_precision.mean()),
        mean_auc=float(daily.auc.mean()),decision_mean_auc=float(actions.auc.mean()),
        decision_mean_ap=float(actions.average_precision.mean()),mean_gain=float(actions.gain.mean()),
        valid_days=int(daily.auc_valid.sum()),
        **{f'count_{a}':int(actions.executed.eq(i).sum()) for i,a in enumerate(ACTIONS)})
    if probabilities is not None:
        result.update({f'pooled_{k}':v for k,v in classification_metrics(labels,probabilities,threshold).items()})
    return result


def choose_policy(policy,agent,state,mask,metrics,rng,auc_reference):
    """Choose one maintenance action using already released diagnostics only."""
    if policy=='dqn':
        return agent.select(state,mask,greedy=True)
    if policy in ACTIONS:
        return ACTIONS.index(policy)
    if policy=='random':
        return int(rng.choice(np.flatnonzero(mask)))
    cov = metrics.get('ks',0.) >= .30 or metrics.get('psi',0.) >= .25
    value = metrics.get('auc',np.nan)
    low = np.isfinite(value) and auc_reference-value >= .03
    action = 3 if cov and low else 1 if cov else 2 if low else 0
    return action if mask[action] else 0


def roll(schedule,policy,assets,name,agent=None,phase='validation',episode=0,export=False):
    """Run one independent frozen policy; never execute unselected alternatives."""
    path = OUT/'predictions'/f'{phase}_{schedule.scenario.iloc[0]}_{policy}.csv' if export else None
    if path is not None and path.exists():
        raise FileExistsError(path)
    before = agent.fingerprint() if agent is not None else None
    env = DailyEnv(schedule,**env_arguments(assets,name),prediction_path=path,
        policy=policy,phase=phase,episode=episode)
    rng = np.random.default_rng(SEED)
    while env.day <= env.days:
        state,mask = env.state(),env.action_mask()
        action = choose_policy(policy,agent,state,mask,env.daily[-1] if env.daily else {},rng,assets['auc_reference'])
        q = agent.q_values(state) if policy=='dqn' else np.full(4,np.nan)
        _,_,done,info = env.step(action)
        info.update({f'q_{a}':float(q[i]) for i,a in enumerate(ACTIONS)})
        if done:
            break
    if agent is not None:
        assert before == agent.fingerprint()
    daily,actions = pd.DataFrame(env.daily),pd.DataFrame(env.actions)
    assert len(actions)==DAYS-WARMUP
    result = summarize_roll(daily,actions,assets['threshold'])
    if path is not None:
        saved = read_csv(path,usecols=['scenario_label','SepsisLabel','probability_post','probability_pre'])
        for suffix,column in [('post','probability_post'),('pre','probability_pre')]:
            result.update({f'pooled_{key}_{suffix}':value for key,value in
                classification_metrics(saved.scenario_label,saved[column],assets['threshold']).items()})
    return result,actions,daily


def select_checkpoint_index(checkpoints):
    """Select the last checkpoint of the best three-validation return window."""
    if len(checkpoints)<SELECT_WINDOW:
        raise ValueError('Need three full validation checkpoints')
    candidates=[]
    for i in range(SELECT_WINDOW-1,len(checkpoints)):
        window=checkpoints[i-SELECT_WINDOW+1:i+1]
        avg=lambda key:float(np.mean([c[key] for c in window]))
        candidates.append((avg('common_score'),avg('decision_mean_ap'),avg('decision_mean_auc'),
            -avg('seconds'),-i,i))
    return max(candidates)[-1]


def train_agent(assets):
    """Train or resume exactly 1000 episodes; pilot uses 30 without claiming FULL."""
    config=COMMON_CONFIG
    reset_seed()
    agent=DQNAgent()
    progress['active_config']=config.config_id
    history,checkpoints,resumed,elapsed=load_training_state(agent,config)
    vals={s:scenario_schedule(assets,'validation',s) for s in SCENARIOS}
    started=time.perf_counter()
    for spec in assets['manifest'].itertuples(index=False):
        ep=int(spec.episode)
        if ep<=resumed:
            continue
        name=f'train_{spec.replay_id}'
        schedule=scenario_schedule(assets,name,spec.scenario,int(spec.onset),float(spec.rate_factor))
        persist_concept_audit(schedule,'train',int(spec.replay_id))
        env=DailyEnv(schedule,**env_arguments(assets,name),phase='train',episode=ep)
        epsilon,lr,loss_start=agent.epsilon,agent.lr,len(agent.losses)
        while env.day<=env.days:
            state=env.state()
            action=agent.select(state,env.action_mask())
            following,reward,done,info=env.step(action)
            agent.learn(state,info['executed'],reward,following,done,env.action_mask())
            if done:
                break
        actions,daily=pd.DataFrame(env.actions),pd.DataFrame(env.daily)
        result=summarize_roll(daily,actions,assets['threshold'])
        losses=agent.losses[loss_start:]
        row=dict(**asdict(config),config_id=config.config_id,episode=ep,scenario=spec.scenario,
            replay_id=int(spec.replay_id),onset=int(spec.onset),rate_factor=float(spec.rate_factor),
            train_return=result['native_return'],train_common_return=result['native_return'],
            mean_auc=result['mean_auc'],mean_daily_ap=result['mean_daily_ap'],mean_gain=result['mean_gain'],
            decision_mean_auc=result['decision_mean_auc'],decision_mean_ap=result['decision_mean_ap'],
            seconds=result['seconds'],cost_relative=result['cost_relative'],epsilon=epsilon,lr=lr,
            updates=agent.updates,loss=float(np.mean(losses)) if losses else np.nan,
            validation_return=np.nan,validation_common_return=np.nan,
            **{f'train_count_{a}':result[f'count_{a}'] for a in ACTIONS})
        agent.end_episode(ep/MAX_EPISODES)
        history.append(row)
        append_table('train_daily_metrics',daily)
        append_table('train_actions',actions)
        if ep%10==0:
            scores=[]
            for scenario,schedule in vals.items():
                score,ac,dy=roll(schedule,'dqn',assets,'validation',agent,episode=ep)
                scores.append(score)
                append_table('validation_history',[score])
                append_table('validation_actions',ac)
                append_table('validation_daily_metrics',dy)
            row['validation_return']=row['validation_common_return']=float(np.mean([s['native_return'] for s in scores]))
            current=dict(episode=ep,common_score=row['validation_return'],
                decision_mean_ap=float(np.mean([s['decision_mean_ap'] for s in scores])),
                decision_mean_auc=float(np.mean([s['decision_mean_auc'] for s in scores])),
                seconds=float(np.mean([s['seconds'] for s in scores])),
                weights={k:v.detach().cpu().clone() for k,v in agent.online.state_dict().items()})
            checkpoints.append(current)
            append_table('controller_freeze',[dict(phase='validation',episode=ep,
                weights_sha256=agent.fingerprint(),passed=True)])
        table('training_history',history)
        checkpoint('training',episodes_done=ep,validation_return=row['validation_return'])
        save_training_state(agent,history,checkpoints,ep,elapsed+time.perf_counter()-started)
        budget_boundary('training_episode')
    assert len(history)==MAX_EPISODES
    index=select_checkpoint_index(checkpoints)
    winner=checkpoints[index]
    window=checkpoints[index-SELECT_WINDOW+1:index+1]
    summary=dict(config_id=config.config_id,**asdict(config),selected_episode=winner['episode'],
        episodes=len(history),selection_score=float(np.mean([c['common_score'] for c in window])),
        selected_native_return=winner['common_score'],updates=agent.updates,
        elapsed_s=elapsed+time.perf_counter()-started)
    torch.save(winner['weights'],OUT/'models'/'selected.pt')
    table('checkpoint_history',[{k:v for k,v in c.items() if k!='weights'} for c in checkpoints])
    table('checkpoint_selection',[summary])
    table('selection_freeze',[dict(**summary,source='mixed_validation_only',before_test=True)])
    RUN_STATE.pop('training',None)
    RUN_STATE['trained']=True
    commit_run('selection_frozen')
    return summary


## plots

In [ ]:
def render_all_from_csv(root):
    """Regenerate scientific figures and smoothing tables without data or model fitting."""
    root=Path(root)
    figures=root/'figures'
    figures.mkdir(exist_ok=True)
    plt.rcParams.update({'figure.dpi':130,'font.size':10,'axes.grid':True,'grid.alpha':.2})
    history=read_csv(root/'training_history.csv')
    actions=read_csv(root/'actions.csv')
    calendar=read_csv(root/'drift_calendar.csv')
    comparisons=read_csv(root/'policy_comparisons.csv')
    manifest=[]
    def save(fig,name):
        """Save a consistently spaced figure with a source-backed manifest entry."""
        fig.tight_layout()
        path=figures/f'{name}.png'
        fig.savefig(path)
        plt.close(fig)
        manifest.append(dict(figure=path.name,source='CSV only'))
    smooth=history[['episode','scenario','train_return','mean_auc','mean_daily_ap','mean_gain']].copy()
    for col in ['train_return','mean_auc','mean_daily_ap','mean_gain']:
        rolling=history[col].rolling(100,min_periods=1)
        smooth[f'{col}_mean100']=rolling.mean()
        smooth[f'{col}_std100']=rolling.std(ddof=0)
        smooth[f'{col}_variance100']=rolling.var(ddof=0)
    smooth.to_csv(root/'training_smoothing.csv',index=False)
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    for ax,col,label in zip(axes.flat,['train_return','mean_auc','mean_daily_ap','mean_gain'],
                            ['Retorno','ROC-AUC post','AP post','Ganancia pre/post']):
        mean=smooth[f'{col}_mean100']
        sd=smooth[f'{col}_std100']
        ax.plot(history.episode,mean,label='Media móvil 100 episodios')
        ax.fill_between(history.episode,mean-sd,mean+sd,alpha=.2,label='±1 DE temporal')
        ax.set(xlabel='Episodio',ylabel=label,title=label)
        ax.legend(fontsize=8)
    save(fig,'01_aprendizaje')
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    scenario_smoothing=[]
    for ax,scenario in zip(axes.flat,SCENARIOS):
        group=history.loc[history.scenario.eq(scenario)].copy()
        group['return_mean15']=group.train_return.rolling(15,min_periods=1).mean()
        scenario_smoothing.append(group[['episode','scenario','train_return','return_mean15']])
        ax.plot(group.episode,group.return_mean15)
        ax.set(title=scenario,xlabel='Episodio',ylabel='Retorno, media móvil 15')
    pd.concat(scenario_smoothing).to_csv(root/'scenario_smoothing.csv',index=False)
    save(fig,'02_retorno_por_escenario')
    counts=actions.groupby(['scenario','policy','action']).size().rename('count').reset_index()
    counts.to_csv(root/'action_counts.csv',index=False)
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        pivot=counts.loc[counts.scenario.eq(scenario)].pivot(index='policy',columns='action',values='count').fillna(0)
        pivot.reindex(columns=ACTIONS,fill_value=0).plot.bar(stacked=True,ax=ax,rot=30)
        ax.set(title=scenario,ylabel='Decisiones')
    save(fig,'03_acciones')
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        group=comparisons.loc[comparisons.scenario.eq(scenario)].set_index('policy')
        group[['decision_mean_auc','decision_mean_ap']].plot.bar(ax=ax,rot=30)
        ax.set(title=scenario,ylabel='Media diaria, días 8–40',ylim=(0,1))
    save(fig,'04_comparacion_politicas')
    dqn=actions.loc[actions.policy.eq('dqn')]
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        group=dqn.loc[dqn.scenario.eq(scenario)]
        for col,label in [('auc_pre','pre'),('auc_post','post')]:
            ax.plot(group.day,group[col],label=label)
        ax.set(title=scenario,xlabel='Día de predicción',ylabel='ROC-AUC')
        ax.legend()
    save(fig,'05_auc_pre_post')
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        group=dqn.loc[dqn.scenario.eq(scenario)]
        for col in ['auc_component','ap_component','gain','cost_penalty','reward']:
            ax.plot(group.day,group[col],label=col)
        ax.axhline(0,color='black',lw=.5)
        ax.set(title=scenario,xlabel='Día de predicción',ylabel='Componente del reward')
        ax.legend(fontsize=8)
    save(fig,'06_componentes_reward')
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        group=dqn.loc[dqn.scenario.eq(scenario)]
        for col in ['ks','psi','cost_relative','gain']:
            ax.plot(group.day,group[col],label=col)
        ax.set(title=scenario,xlabel='Día de predicción',ylabel='Valor observado')
        ax.legend(fontsize=8)
    save(fig,'07_monitoreo')
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        group=calendar.loc[calendar.scenario.eq(scenario)]
        for col in ['original_prevalence','scenario_prevalence','flip_rate']:
            ax.plot(group.day,group[col],label=col)
        ax.axvline(8,color='black',ls=':',label='Primera decisión')
        ax.set(title=scenario,xlabel='Día',ylabel='Proporción')
        ax.legend(fontsize=8)
    save(fig,'08_deriva_etiquetas')
    fig,axes=plt.subplots(1,2,figsize=(13,4))
    axes[0].plot(history.episode,history.epsilon)
    axes[0].set(xlabel='Episodio',ylabel='Epsilon')
    axes[1].plot(history.episode,history.loss.rolling(100,min_periods=1).mean())
    axes[1].set(xlabel='Episodio',ylabel='Huber, media móvil 100')
    save(fig,'09_exploracion_huber')
    samples=read_csv(root/'drift_samples.csv')
    fig,axes=plt.subplots(4,2,figsize=(13,13))
    for ax,vital in zip(axes.flat,VITALS):
        for day in (1,40):
            values=samples.loc[samples.scenario.eq('covariate') & samples.day.eq(day)
                & samples.vital.eq(vital),'value'].dropna()
            ax.hist(values,bins=45,density=True,histtype='step',label=f'Día {day}')
        ax.set(title=vital,xlabel='Valor',ylabel='Densidad')
        ax.legend()
    axes.flat[-1].set_visible(False)
    save(fig,'10_deriva_vitales')
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    for ax,scenario in zip(axes.flat,SCENARIOS):
        for policy in ('dqn','A1'):
            group=actions.loc[actions.scenario.eq(scenario)&actions.policy.eq(policy)]
            ax.plot(group.day,group.auc_post,label=policy)
        ax.set(title=scenario,xlabel='Día de predicción',ylabel='ROC-AUC post')
        ax.legend()
    save(fig,'11_ddqn_frente_a1')
    pd.DataFrame(manifest).to_csv(root/'figure_manifest.csv',index=False)
    return manifest


## orchestration

In [ ]:
def numerical_contracts():
    """Check gain signs, invalid days and nonduplicated A1 rewards before fitting."""
    before=dict(auc_valid=True,auc=.7,average_precision=.4)
    post=dict(auc_valid=True,auc=.8,average_precision=.5)
    checks=[]
    def check(name,passed):
        """Record an integrity claim only after its condition has been evaluated."""
        assert passed,name
        checks.append(dict(check=name,passed=True))
    positive=reward_components(post,before,2.,4.,COMMON_CONFIG)
    negative=reward_components(before,post,2.,4.,COMMON_CONFIG)
    same=reward_components(before,before,0.,4.,COMMON_CONFIG)
    invalid=reward_components({**post,'auc_valid':False},before,2.,4.,COMMON_CONFIG)
    check('positive_gain',np.isclose(positive['gain'],.2))
    check('negative_gain_positive_reward',negative['gain']<0 and negative['reward']>0)
    check('a1_zero_gain',same['gain']==0.)
    check('invalid_cost_only',np.isnan(invalid['gain']) and invalid['reward']==-.0125)
    check('absolute_gain_cost_arithmetic',np.isclose(positive['reward'],1.4875))
    table('numerical_contracts',checks)


def verify_exports(assets):
    """Rebuild paired test metrics and verify decision counts, isolation and chronology."""
    checks=[]
    def check(name,passed):
        """Fail the run rather than publish inconsistent scientific artifacts."""
        assert passed,name
        checks.append(dict(check=name,passed=True))
    train=read_csv(OUT/'train_actions.csv')
    history=read_csv(OUT/'training_history.csv')
    check('training_episodes',history.episode.to_list()==list(range(1,MAX_EPISODES+1)))
    check('training_decisions',len(train)==MAX_EPISODES*(DAYS-WARMUP))
    check('no_duplicate_train_decisions',not train.duplicated(['episode','day']).any())
    check('33_decisions_per_episode',train.groupby('episode').size().eq(33).all())
    for filename in ('train_actions','validation_actions','actions'):
        actions=read_csv(OUT/f'{filename}.csv')
        check(f'{filename}_calendar',actions.day.between(8,40).all() and
              actions.label_received_day.eq(actions.day+1).all())
        check(f'{filename}_seven_day_window',actions.training_min_day.eq(actions.day-7).all() and
              actions.training_max_day.eq(actions.day-1).all() and
              actions.training_label_available_max.le(actions.day).all())
        valid=actions.gain_evaluable
        np.testing.assert_allclose(actions.loc[valid,'gain'],
            actions.loc[valid,'delta_auc']+actions.loc[valid,'delta_ap'],rtol=0,atol=1e-12)
        expected=actions.auc_component+actions.ap_component+actions.gain.fillna(0)-actions.cost_penalty
        np.testing.assert_allclose(actions.reward,expected,rtol=0,atol=1e-12)
        check(f'{filename}_reward_arithmetic',True)
        check(f'{filename}_a1_gain',actions.loc[valid&actions.action.eq('A1'),'gain'].eq(0).all())
        check(f'{filename}_invalid_gain',actions.loc[~valid,'gain'].isna().all())
        np.testing.assert_allclose(actions.loc[~valid,'reward'],-actions.loc[~valid,'cost_penalty'],atol=1e-12)
    actions=read_csv(OUT/'actions.csv')
    for scenario in SCENARIOS:
        expected_ids=None
        for policy in ('dqn',*ACTIONS,'random','heuristic'):
            path=OUT/'predictions'/f'test_{scenario}_{policy}.csv'
            saved=read_csv(path)
            ids=saved[['patient_id','ICULOS','hour','bed','day']]
            if expected_ids is None:
                expected_ids=ids
            else:
                pd.testing.assert_frame_equal(ids,expected_ids)
            check(f'{scenario}_{policy}_records',len(saved)==33*24*BEDS and
                not saved.duplicated(['patient_id','ICULOS']).any())
            chosen=actions.loc[actions.scenario.eq(scenario)&actions.policy.eq(policy)]
            for day,rows in saved.groupby('day'):
                row=chosen.loc[chosen.day.eq(day)].iloc[0]
                for suffix in ('pre','post'):
                    metrics=classification_metrics(rows.scenario_label,rows[f'probability_{suffix}'],assets['threshold'])
                    for key in ('auc','average_precision','f1_positive','f1_macro','precision','recall','pr_auc_trapz'):
                        np.testing.assert_allclose(metrics[key],row[f'{key}_{suffix}'],rtol=0,atol=1e-12,equal_nan=True)
                if row.action=='A1':
                    np.testing.assert_array_equal(rows.probability_pre,rows.probability_post)
            check(f'{scenario}_{policy}_metrics_roundtrip',True)
    table('integrity_checks',checks)
    return checks


def run_experiment():
    """Prepare, train, freeze selection, evaluate seven policies and verify exports."""
    initialize_run()
    if RUN_STATE.get('complete'):
        print('This run is already complete; use plots to regenerate figures.')
        return
    numerical_contracts()
    table('reward_configs',[dict(**asdict(COMMON_CONFIG),config_id=COMMON_CONFIG.config_id,
        seed=SEED,episodes=MAX_EPISODES)])
    table('provenance',[dict(**RUN_STATE['identity'],gpu=torch.cuda.get_device_name(0),segment=SEGMENT_VERSION)])
    assets=prepare_shared()
    checkpoint('prepared',calibration_qualified=assets['calibration_qualified'],
        cost_unit_s=assets['cost_unit'],cov_rate=assets['cov_rate'],concept_rate=assets['concept_rate'])
    if not RUN_STATE.get('trained'):
        train_agent(assets)
    budget_boundary('before_test')
    schedules={s:scenario_schedule(assets,'test',s) for s in SCENARIOS}
    export_drift_tables(schedules)
    agent=DQNAgent()
    weights=torch.load(OUT/'models'/'selected.pt',map_location=DEVICE,weights_only=True)
    agent.online.load_state_dict(weights)
    agent.target.load_state_dict(weights)
    results=read_csv(OUT/'policy_comparisons.csv').to_dict('records') if RUN_STATE.get('test_done') else []
    for scenario,schedule in schedules.items():
        persist_concept_audit(schedule,'test','mixed_test')
        for policy in ('dqn',*ACTIONS,'random','heuristic'):
            key=f'{scenario}_{policy}'
            if key in RUN_STATE.get('test_done',[]):
                continue
            budget_boundary('before_test_policy')
            result,actions,daily=roll(schedule,policy,assets,'test',agent if policy=='dqn' else None,
                phase='test',export=True)
            results.append(result)
            append_table('actions',actions)
            append_table('daily_metrics',daily)
            table('policy_comparisons',results)
            RUN_STATE.setdefault('test_done',[]).append(key)
            checkpoint('test',completed_test_policies=len(RUN_STATE['test_done']))
            commit_run('test_policy')
    checks=verify_exports(assets)
    figures=render_all_from_csv(OUT)
    table('artifact_manifest',[dict(path=p.relative_to(OUT).as_posix(),bytes=p.stat().st_size)
        for p in sorted(OUT.rglob('*')) if p.is_file()])
    checkpoint('complete',status='pilot_complete' if MODE=='pilot' else 'complete',
        total_episodes=MAX_EPISODES,total_training_decisions=MAX_EPISODES*33,
        integrity_passed=all(c['passed'] for c in checks),calibration_qualified=assets['calibration_qualified'],
        elapsed_s=time.perf_counter()-RUN_STARTED,figure_files=len(figures))
    RUN_STATE['complete']=True
    commit_run('complete')


if __name__=='__main__':
    if MODE=='plots':
        render_all_from_csv(OUT)
    else:
        try:
            run_experiment()
        except SegmentYield as exc:
            checkpoint('partial',status='partial',reason=str(exc))
            print('Durable snapshot available; mount this kernel as the sole predecessor.',flush=True)
        except Exception:
            checkpoint('failed',status='failed',traceback=traceback.format_exc())
            print('Run failed; see run_summary.json.', flush=True)
